# Robustness checks

This notebook idea is to provide multiple robustness checks to support the claim \textit{Do oil volatility shocks matter more for oil exporters nations compared to non-oil exporters}

### Set up panel data:

In [5]:
import pandas as pd
import datetime
import numpy as np
from linearmodels.panel import PanelOLS
import matplotlib.pyplot as plt
from itertools import combinations

In [6]:
CDS_data = pd.read_csv('data/processed/CDS/Weekly_CDS.csv')
CDS_data['Date'] = pd.to_datetime(CDS_data['Date'])
CDS_data.set_index('Date', inplace=True)

Oil_data = pd.read_csv('data/processed/Oil/oil_prices_datastream.csv')
Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
Oil_data.set_index('Date', inplace=True)

Macro_risk_variables = pd.read_csv('data/processed/Macroeconomic_variables/macro_risk_variables.csv')
Macro_risk_variables['Date'] = pd.to_datetime(Macro_risk_variables['Date'])
Macro_risk_variables.set_index('Date', inplace=True)

VIX = pd.read_csv('data/processed/Macroeconomic_variables/VIXCLS.csv')
VIX['Date'] = pd.to_datetime(VIX['Date'])
VIX.set_index('Date', inplace=True)

OVX = pd.read_csv('data/processed/Macroeconomic_variables/OVXCLS.csv')
OVX['Date'] = pd.to_datetime(OVX['Date'])
OVX.set_index('Date', inplace=True)


# =============================================================================
# CDS groups
# =============================================================================

CDS_universe = ['Abu Dhabi', 'Argentina', 'Australia','Austria','Bahrain', 'Belgium', 'Brazil', 'Bulgaria', 'Chile', 'China', 'Colombia','Costa Rica', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Dominican Republic',
'Dubai', 'Egypt', 'El Salvador', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Hungary', 'India', 'Indonesia', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Kazakhstan',
'Kuwait', 'Latvia', 'Lithuania', 'Malaysia', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Peru', 'Philippines', 'Poland', 'Qatar', 'Romania', 'Saudi Arabia',
'Serbia', 'Slovakia', 'Slovenia', 'South Africa','South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Thailand', 'Trinidad and Tobago', 'Turkey', 'United Kingdom', 'United States', 'Uruguay','Vietnam']


# =============================================================================
# MERGE (WIDE FORMAT)
# =============================================================================

merged = CDS_data.copy()
merged = pd.merge(merged, Oil_data, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, Macro_risk_variables, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, VIX, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, OVX, left_index=True, right_index=True, how='inner')

# Filter date range
merged = merged[merged.index >= datetime.datetime(2014, 1, 1)].copy()
merged.reset_index(inplace=True)


# Oil returns (percent change)
merged['Oil_ret'] = merged['Brent'].pct_change()

# OVX change (percent change - it's a volatility index)
merged['OVX_chg'] = merged['OVXCLS'].pct_change()

# VIX change (percent change)
merged['VIX_chg'] = merged['VIXCLS'].pct_change()

# DXY change (percent change)
merged['DXY_chg'] = merged['DXY'].pct_change()

# Treasury change (diff change)
merged['UST2Y_chg'] = merged['UST2Y'].diff()
merged['UST5Y_chg'] = merged['UST5Y'].diff()
merged['UST10Y_chg'] = merged['UST10Y'].diff()

merged = merged.reset_index()

global_vars = ['Date', 'Brent', 'Oil_ret', 'OVXCLS', 'OVX_chg', 
               'VIXCLS', 'VIX_chg', 'DXY_chg', 'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']

panel_data = []

for country in CDS_universe:
    if country not in merged.columns:
        print(f"Warning: {country} not in CDS data")
        continue
        
    temp = merged[global_vars + [country]].copy()
    temp['Country'] = country
    temp['CDS'] = temp[country]
    temp = temp.drop(columns=[country])
    panel_data.append(temp)

panel = pd.concat(panel_data, ignore_index=True)
panel = panel.sort_values(by=['Country', 'Date']).reset_index(drop=True)

# =============================================================================
# COMPUTE COUNTRY-SPECIFIC VARIABLES (CDS returns)
# =============================================================================

# CDS returns - THIS one needs groupby because it's country-specific
panel['CDS_ret'] = panel.groupby('Country')['CDS'].pct_change()
# Drop NaN (first observation per country)
panel = panel.dropna(subset=['CDS_ret', 'OVX_chg', 'VIX_chg', 'DXY_chg', 'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']).reset_index(drop=True)

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_43015/1880135811.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_43015/1880135811.py:10: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  Macro_risk_variables['Date'] = pd.to_datetime(Macro_risk_variables['Date'])


## Robustness test 1: Permutation test

In [7]:
from itertools import combinations
from linearmodels.panel import PanelOLS
import pandas as pd
import numpy as np

# =============================================================================
# 1. FILTER TO MSCI EM SAMPLE (17 countries)
# =============================================================================

oil_exporters = ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 
                 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
controls = ['Indonesia', 'Philippines', 'Turkey', 'Chile', 'China', 
            'South Africa', 'South Korea', 'Thailand']
sample_countries = oil_exporters + controls

panel_sample = panel[panel['Country'].isin(sample_countries)].copy()
print(f"Panel shape: {panel_sample.shape}")
print(f"Countries: {panel_sample['Country'].nunique()}")
print(f"Weeks: {panel_sample['Date'].nunique()}")

# =============================================================================
# 2. ENUMERATE ALL POSSIBLE TREATMENT GROUPS
# =============================================================================

# All possible non-empty, non-full subsets = 2^17 - 2 = 131,070
# Due to symmetry (treatment vs control is arbitrary), we only need k = 1 to 8
# This gives us ~65,000 unique splits

all_countries = list(panel_sample['Country'].unique())
n_countries = len(all_countries)

print(f"\nTotal countries: {n_countries}")

# Generate all possible treatment groups of size 1 to n-1
# For efficiency, only go up to n//2 (due to symmetry)
all_treatment_groups = []
for k in range(1, n_countries // 2 + 1):
    for combo in combinations(all_countries, k):
        all_treatment_groups.append(set(combo))

# Add size n//2 + 1 to n-1 only if we want the full enumeration
# (the interaction coefficient just flips sign, so it's redundant for ranking)

print(f"Total unique splits (k=1 to {n_countries//2}): {len(all_treatment_groups):,}")

# =============================================================================
# 3. DEFINE REGRESSION FUNCTION
# =============================================================================

def run_interaction_regression(panel_df, treatment_countries, threshold_pct=1.0):
    """
    Run panel regression with OVX × Treatment interaction.
    
    Parameters:
    - panel_df: panel data
    - treatment_countries: set of country names in treatment group
    - threshold_pct: 1.0 for full sample, 0.10 for top 10%, etc.
    
    Returns:
    - beta: interaction coefficient
    - pval: p-value
    - n_obs: number of observations
    """
    temp = panel_df.copy()
    
    # Create treatment indicator
    temp['Treatment'] = temp['Country'].isin(treatment_countries).astype(int)
    temp['OVX_x_Treatment'] = temp['OVX_chg'] * temp['Treatment']
    
    # Filter to threshold
    if threshold_pct < 1.0:
        oil_qtl = temp['OVX_chg'].quantile(1 - threshold_pct)
        temp = temp[temp['OVX_chg'] > oil_qtl].copy()
    
    if len(temp) < 50:
        return np.nan, np.nan, len(temp)
    
    # Prepare for regression
    temp['Date'] = pd.to_datetime(temp['Date'])
    reg_data = temp.set_index(['Country', 'Date'])
    
    y = reg_data['CDS_ret']
    X = reg_data[['OVX_chg', 'OVX_x_Treatment', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
    
    try:
        model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True)
        res = model.fit(cov_type='clustered', cluster_entity=True)
        return res.params['OVX_x_Treatment'], res.pvalues['OVX_x_Treatment'], len(temp)
    except:
        return np.nan, np.nan, len(temp)

# =============================================================================
# 4. RUN PERMUTATION TEST FOR ALL SPLITS
# =============================================================================

thresholds = {
    'Full': 1.0,
    'Top 10%': 0.10,
    'Top 5%': 0.05,
    'Top 1%': 0.01
}

# Store results
permutation_results = {thresh: [] for thresh in thresholds.keys()}

# True classification
true_treatment = set(oil_exporters)

print(f"\nRunning {len(all_treatment_groups):,} permutations for each threshold...")
print("This may take a few minutes...\n")

for i, treatment_group in enumerate(all_treatment_groups):
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1:,} / {len(all_treatment_groups):,}")
    
    for thresh_name, thresh_pct in thresholds.items():
        beta, pval, n_obs = run_interaction_regression(panel_sample, treatment_group, thresh_pct)
        permutation_results[thresh_name].append({
            'treatment_group': treatment_group,
            'treatment_size': len(treatment_group),
            'beta': beta,
            'pval': pval,
            'n_obs': n_obs,
            'is_true': treatment_group == true_treatment
        })

print("Done!")

# =============================================================================
# 5. COMPUTE TRUE CLASSIFICATION RESULTS
# =============================================================================

print("\n" + "="*70)
print("TRUE OIL EXPORTER CLASSIFICATION RESULTS")
print("="*70)

true_results = {}
for thresh_name, thresh_pct in thresholds.items():
    beta, pval, n_obs = run_interaction_regression(panel_sample, true_treatment, thresh_pct)
    true_results[thresh_name] = {'beta': beta, 'pval': pval, 'n_obs': n_obs}
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    print(f"{thresh_name:>10}: β = {beta:>8.4f}, p = {pval:.4f} {sig}, N = {n_obs}")

# =============================================================================
# 6. COMPUTE PERCENTILE RANKS
# =============================================================================

print("\n" + "="*70)
print("PERCENTILE RANK OF TRUE CLASSIFICATION (vs all possible splits)")
print("="*70)

for thresh_name in thresholds.keys():
    df = pd.DataFrame(permutation_results[thresh_name])
    df = df.dropna(subset=['beta'])
    
    true_beta = true_results[thresh_name]['beta']
    
    # Percentile rank (% of permutations with beta <= true_beta)
    percentile = (df['beta'] <= true_beta).mean() * 100
    
    # Permutation p-value (% of permutations with |beta| >= |true_beta|)
    perm_pval = (df['beta'].abs() >= abs(true_beta)).mean()
    
    print(f"{thresh_name:>10}: Percentile = {percentile:>6.2f}%, Perm p-value = {perm_pval:.4f}, N permutations = {len(df):,}")

# =============================================================================
# 7. DISTRIBUTION BY TREATMENT GROUP SIZE
# =============================================================================

print("\n" + "="*70)
print("BETA DISTRIBUTION BY TREATMENT GROUP SIZE (Top 10%)")
print("="*70)

df_10 = pd.DataFrame(permutation_results['Top 10%']).dropna(subset=['beta'])
size_summary = df_10.groupby('treatment_size')['beta'].agg(['mean', 'std', 'min', 'max', 'count'])
print(size_summary.round(4))

print(f"\nTrue classification: {len(true_treatment)} countries, β = {true_results['Top 10%']['beta']:.4f}")

Panel shape: (9129, 14)
Countries: 17
Weeks: 537

Total countries: 17
Total unique splits (k=1 to 8): 65,535

Running 65,535 permutations for each threshold...
This may take a few minutes...

  Processed 10,000 / 65,535
  Processed 20,000 / 65,535
  Processed 30,000 / 65,535
  Processed 40,000 / 65,535
  Processed 50,000 / 65,535
  Processed 60,000 / 65,535
Done!

TRUE OIL EXPORTER CLASSIFICATION RESULTS
      Full: β =   0.0253, p = 0.3322 , N = 9129
   Top 10%: β =   0.1412, p = 0.0013 ***, N = 901
    Top 5%: β =   0.1473, p = 0.0261 **, N = 442
    Top 1%: β =   0.6082, p = 0.0584 *, N = 85

PERCENTILE RANK OF TRUE CLASSIFICATION (vs all possible splits)
      Full: Percentile =  80.52%, Perm p-value = 0.3920, N permutations = 65,535
   Top 10%: Percentile =  99.29%, Perm p-value = 0.0137, N permutations = 65,535
    Top 5%: Percentile =  96.49%, Perm p-value = 0.0667, N permutations = 65,535
    Top 1%: Percentile =  95.10%, Perm p-value = 0.0929, N permutations = 65,535

BETA DIS

In [8]:
import json

for thresh_name, results_list in permutation_results.items():
    df = pd.DataFrame(results_list)
    
    # Convert set to string for CSV storage
    df['treatment_group'] = df['treatment_group'].apply(lambda x: ','.join(sorted(x)))
    
    # Clean filename
    filename = f"permutation_results_{thresh_name.replace(' ', '_').replace('%', 'pct')}.csv"
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} ({len(df):,} rows)")

# Also save true results
true_df = pd.DataFrame(true_results).T
true_df.to_csv('permutation_true_results.csv')
print("Saved: permutation_true_results.csv")

Saved: permutation_results_Full.csv (65,535 rows)
Saved: permutation_results_Top_10pct.csv (65,535 rows)
Saved: permutation_results_Top_5pct.csv (65,535 rows)
Saved: permutation_results_Top_1pct.csv (65,535 rows)
Saved: permutation_true_results.csv


## Robustness check 2: Placebo groups

Comparing other discrimination criteria in both the original vs extended group of countries. See if the oil effect persists or if it is diluted.

In [10]:
# =============================================================================
# 1. FILTER TO MSCI EM SAMPLE (17 countries)
# =============================================================================

oil_exporters = ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 
                 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
controls = ['Indonesia', 'Philippines', 'Turkey', 'Chile', 'China', 
            'South Africa', 'South Korea', 'Thailand']
sample_countries = oil_exporters + controls

panel_sample = panel[panel['Country'].isin(sample_countries)].copy()
print(f"Panel shape: {panel_sample.shape}")
print(f"Countries: {panel_sample['Country'].nunique()}")
print(f"Weeks: {panel_sample['Date'].nunique()}")

# =============================================================================
# 2. DEFINE ALL CLASSIFICATIONS
# =============================================================================

classifications = {
    'Oil Exporters': {
        'treatment': ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia'],
        'control': ['Indonesia', 'Philippines', 'Turkey', 'Chile', 'China', 'South Africa', 'South Korea', 'Thailand']
    },
    'GCC': {
        'treatment': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia'],
        'control': ['Egypt', 'Brazil', 'South Korea', 'China', 'South Africa', 'Colombia', 'Mexico', 'Malaysia', 'Chile', 'Turkey', 'Indonesia', 'Philippines', 'Thailand']
    },
    'Low GDP pc': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea', 'Saudi Arabia', 'Chile', 'Turkey', 'Malaysia', 'Mexico', 'China'],
        'treatment': ['Philippines', 'Egypt', 'Indonesia', 'South Africa', 'Colombia', 'Brazil', 'Thailand']
    },
    'High Debt/GDP': {
        'treatment': ['Egypt', 'Brazil', 'China', 'Malaysia', 'South Africa', 'Colombia', 'Mexico', 'Qatar'],
        'control': ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Chile', 'Turkey', 'Indonesia', 'South Korea', 'Philippines', 'Thailand']
    },
    'Low CA/GDP': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea', 'Malaysia', 'Saudi Arabia', 'China', 'Philippines', 'Mexico'],
        'treatment': ['Colombia', 'Chile', 'Egypt', 'Brazil', 'Turkey', 'South Africa', 'Indonesia', 'Thailand']
    },
    'Low Fiscal Bal': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea'],
        'treatment': ['Egypt', 'Brazil', 'Saudi Arabia', 'China', 'South Africa', 'Colombia', 'Mexico', 'Malaysia', 'Chile', 'Turkey', 'Indonesia', 'Philippines', 'Thailand']
    },
    'Asian': {
        'treatment': ['South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand'],
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'Brazil', 'Colombia', 'Mexico', 'Chile', 'South Africa', 'Turkey']
    },
    'Latin America': {
        'treatment': ['Brazil', 'Colombia', 'Mexico', 'Chile'],
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand', 'South Africa', 'Turkey']
    },
    'Low Credit Rating': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'South Korea', 'Chile', 'China', 'Malaysia'],
        'treatment': ['Egypt', 'Brazil', 'Colombia', 'Mexico', 'Indonesia', 'Philippines', 'Turkey', 'South Africa', 'Thailand']
    },
}

# Thresholds to test
thresholds = [1.0, 0.10, 0.05, 0.01]  # 1.0 = full sample

# =============================================================================
# 3. RUN ALL REGRESSIONS
# =============================================================================

results_list = []

for class_name, groups in classifications.items():
    
    treatment = groups['treatment']
    control = groups['control']
    
    # Create classification-specific panel (using filtered sample)
    temp_panel = panel_sample.copy()
    temp_panel['Treatment'] = temp_panel['Country'].isin(treatment).astype(int)
    temp_panel['OVX_x_Treatment'] = temp_panel['OVX_chg'] * temp_panel['Treatment']
    
    for pct in thresholds:
        
        # Filter to threshold
        if pct == 1.0:
            subset = temp_panel.copy()
            threshold_label = 'Full'
        else:
            oil_qtl = temp_panel['OVX_chg'].quantile(1 - pct)
            subset = temp_panel[temp_panel['OVX_chg'] > oil_qtl].copy()
            threshold_label = f'Top {pct*100:.0f}%'
        
        # Skip if too few observations
        if len(subset) < 50:
            continue
        
        # Prepare for linearmodels
        subset['Date'] = pd.to_datetime(subset['Date'])
        reg_data = subset.set_index(['Country', 'Date'])
        
        y = reg_data['CDS_ret']
        
        # Model A: Country FE + explicit controls
        try:
            X_a = reg_data[['OVX_chg', 'OVX_x_Treatment', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
            model_a = PanelOLS(y, X_a, entity_effects=True, time_effects=False, drop_absorbed=True)
            res_a = model_a.fit(cov_type='clustered', cluster_entity=True)
            beta_a = res_a.params['OVX_x_Treatment']
            pval_a = res_a.pvalues['OVX_x_Treatment']
        except:
            beta_a, pval_a = np.nan, np.nan
        
        # Model B: Country FE + Time FE
        try:
            X_b = reg_data[['OVX_x_Treatment']]
            model_b = PanelOLS(y, X_b, entity_effects=True, time_effects=True, drop_absorbed=True)
            res_b = model_b.fit(cov_type='clustered', cluster_entity=True)
            beta_b = res_b.params['OVX_x_Treatment']
            pval_b = res_b.pvalues['OVX_x_Treatment']
        except:
            beta_b, pval_b = np.nan, np.nan
        
        results_list.append({
            'Classification': class_name,
            'Threshold': threshold_label,
            'N': len(subset),
            'N_treatment': subset['Treatment'].sum(),
            'N_control': len(subset) - subset['Treatment'].sum(),
            'Beta_A': beta_a,
            'pval_A': pval_a,
            'Beta_B': beta_b,
            'pval_B': pval_b,
        })

# =============================================================================
# 4. CREATE RESULTS DATAFRAME
# =============================================================================

results_df = pd.DataFrame(results_list)

# =============================================================================
# 5. PRINT RESULTS - MODEL A
# =============================================================================

print("\n" + "="*90)
print("MODEL A: Country FE + Global Controls (VIX, DXY, UST10Y)")
print("="*90)

# Pivot for easier reading
pivot_a = results_df.pivot(index='Classification', columns='Threshold', values='Beta_A')
pivot_a = pivot_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]  # Order columns

pivot_pval_a = results_df.pivot(index='Classification', columns='Threshold', values='pval_A')
pivot_pval_a = pivot_pval_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

# Print with significance stars
print(f"\n{'Classification':<20} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*70)

for idx in pivot_a.index:
    row_str = f"{idx:<20}"
    for col in pivot_a.columns:
        beta = pivot_a.loc[idx, col]
        pval = pivot_pval_a.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# 6. PRINT RESULTS - MODEL B
# =============================================================================

print("\n" + "="*90)
print("MODEL B: Country FE + Time FE")
print("="*90)

pivot_b = results_df.pivot(index='Classification', columns='Threshold', values='Beta_B')
pivot_b = pivot_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

pivot_pval_b = results_df.pivot(index='Classification', columns='Threshold', values='pval_B')
pivot_pval_b = pivot_pval_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

print(f"\n{'Classification':<20} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*70)

for idx in pivot_b.index:
    row_str = f"{idx:<20}"
    for col in pivot_b.columns:
        beta = pivot_b.loc[idx, col]
        pval = pivot_pval_b.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# 7. HIGHLIGHT: COMPARE OIL EXPORTERS VS OTHER CLASSIFICATIONS
# =============================================================================

print("\n" + "="*90)
print("COMPARISON: Oil Exporters vs Placebo Classifications (Top 10% OVX, Model A)")
print("="*90)

top10_results = results_df[results_df['Threshold'] == 'Top 10%'].copy()
top10_results = top10_results.sort_values('Beta_A', ascending=False)

print(f"\n{'Classification':<20} {'Beta':>10} {'p-value':>10} {'Significant':>12}")
print("-"*55)

for _, row in top10_results.iterrows():
    sig = '***' if row['pval_A'] < 0.01 else '**' if row['pval_A'] < 0.05 else '*' if row['pval_A'] < 0.1 else ''
    is_sig = 'Yes' if row['pval_A'] < 0.05 else 'No'
    print(f"{row['Classification']:<20} {row['Beta_A']:>10.3f} {row['pval_A']:>10.3f} {is_sig:>10} {sig}")

# =============================================================================
# 8. SAVE RESULTS
# =============================================================================

results_df.to_csv('placebo_classification_results.csv', index=False)
print("\nSaved: placebo_classification_results.csv")

Panel shape: (9129, 14)
Countries: 17
Weeks: 537

MODEL A: Country FE + Global Controls (VIX, DXY, UST10Y)

Classification               Full      Top 10%       Top 5%       Top 1%
----------------------------------------------------------------------
Asian                    0.015      -0.021       0.027       0.477  
GCC                     -0.040*     -0.004      -0.088       0.100  
High Debt/GDP            0.041       0.078       0.115       0.308  
Latin America            0.083***     0.140***     0.173**    -0.201  
Low CA/GDP              -0.002      -0.016      -0.006      -0.683**
Low Credit Rating        0.018       0.038       0.087      -0.309  
Low Fiscal Bal           0.042**     0.042       0.107*      0.104  
Low GDP pc               0.021       0.051       0.091      -0.114  
Oil Exporters            0.025       0.141***     0.147**     0.608* 

* p<0.1, ** p<0.05, *** p<0.01

MODEL B: Country FE + Time FE

Classification               Full      Top 10%       Top 5% 

## Extended sample

In [12]:
from linearmodels.panel import PanelOLS
import pandas as pd
import numpy as np

# =============================================================================
# 1. USE FULL PANEL (67 countries)
# =============================================================================

print(f"Full panel shape: {panel.shape}")
print(f"Countries: {panel['Country'].nunique()}")
print(f"Weeks: {panel['Date'].nunique()}")

# List all countries in panel
all_countries = panel['Country'].unique().tolist()
print(f"\nCountries in panel:\n{sorted(all_countries)}")

# =============================================================================
# 2. DEFINE EXPANDED CLASSIFICATIONS
# =============================================================================

# Oil exporters (net oil exporters from the 67-country universe)
oil_exporters_expanded = [
    # GCC
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Kuwait', 'Bahrain',
    # Other Middle East
    'Iraq',
    # Latin America
    'Colombia', 'Mexico', 'Brazil', 'Ecuador', 'Venezuela', 'Trinidad and Tobago',
    # Africa
    'Egypt', 'Nigeria',
    # Europe
    'Norway', 'Kazakhstan',
    # Asia
    'Malaysia',
]

# Filter to countries actually in panel
oil_exporters_expanded = [c for c in oil_exporters_expanded if c in all_countries]
controls_expanded = [c for c in all_countries if c not in oil_exporters_expanded]

print(f"\nOil exporters in panel ({len(oil_exporters_expanded)}): {oil_exporters_expanded}")
print(f"Controls in panel ({len(controls_expanded)}): {controls_expanded}")

# =============================================================================
# 3. DEFINE ALL CLASSIFICATIONS
# =============================================================================

classifications = {
    'Oil Exporters': {
        'treatment': oil_exporters_expanded,
        'control': controls_expanded
    },
    'GCC': {
        'treatment': [c for c in ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Kuwait', 'Bahrain'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Kuwait', 'Bahrain']]
    },
    'Middle East': {
        'treatment': [c for c in ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Kuwait', 'Bahrain', 'Iraq', 'Israel', 'Egypt'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Kuwait', 'Bahrain', 'Iraq', 'Israel', 'Egypt']]
    },
    'Latin America': {
        'treatment': [c for c in ['Brazil', 'Colombia', 'Mexico', 'Chile', 'Argentina', 'Peru', 'Panama', 
                                   'Costa Rica', 'Dominican Republic', 'El Salvador', 'Jamaica', 'Uruguay'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Brazil', 'Colombia', 'Mexico', 'Chile', 'Argentina', 'Peru', 'Panama',
                                                           'Costa Rica', 'Dominican Republic', 'El Salvador', 'Jamaica', 'Uruguay']]
    },
    'Asia (ex-Middle East)': {
        'treatment': [c for c in ['China', 'South Korea', 'Japan', 'India', 'Indonesia', 'Malaysia', 'Philippines', 
                                   'Thailand', 'Vietnam', 'Hong Kong', 'Sri Lanka'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['China', 'South Korea', 'Japan', 'India', 'Indonesia', 'Malaysia', 
                                                           'Philippines', 'Thailand', 'Vietnam', 'Hong Kong', 'Sri Lanka']]
    },
    'Emerging Europe': {
        'treatment': [c for c in ['Turkey', 'Poland', 'Hungary', 'Czechia', 'Romania', 'Bulgaria', 'Croatia', 
                                   'Serbia', 'Slovakia', 'Slovenia', 'Estonia', 'Latvia', 'Lithuania'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Turkey', 'Poland', 'Hungary', 'Czechia', 'Romania', 'Bulgaria', 
                                                           'Croatia', 'Serbia', 'Slovakia', 'Slovenia', 'Estonia', 'Latvia', 'Lithuania']]
    },
    'Developed Markets': {
        'treatment': [c for c in ['United States', 'United Kingdom', 'Germany', 'France', 'Japan', 'Australia', 
                                   'New Zealand', 'Norway', 'Sweden', 'Denmark', 'Finland', 'Netherlands', 
                                   'Belgium', 'Austria', 'Ireland', 'Spain', 'Italy'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['United States', 'United Kingdom', 'Germany', 'France', 'Japan', 
                                                           'Australia', 'New Zealand', 'Norway', 'Sweden', 'Denmark', 'Finland', 
                                                           'Netherlands', 'Belgium', 'Austria', 'Ireland', 'Spain', 'Italy']]
    },
    'High Yield / Distressed': {
        'treatment': [c for c in ['Argentina', 'Egypt', 'Sri Lanka', 'El Salvador', 'Jamaica', 'Iraq', 
                                   'Vietnam', 'Dominican Republic'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Argentina', 'Egypt', 'Sri Lanka', 'El Salvador', 'Jamaica', 
                                                           'Iraq', 'Vietnam', 'Dominican Republic']]
    },
    'BRICS': {
        'treatment': [c for c in ['Brazil', 'China', 'India', 'South Africa'] if c in all_countries],
        'control': [c for c in all_countries if c not in ['Brazil', 'China', 'India', 'South Africa']]
    },
    
    'Non-GCC Oil Exporters': {
    'treatment': ['Colombia', 'Mexico', 'Brazil', 'Trinidad and Tobago', 'Egypt', 
                  'Norway', 'Kazakhstan', 'Malaysia', 'Iraq'],
    'control': [c for c in all_countries if c not in ['Colombia', 'Mexico', 'Brazil', 
                'Trinidad and Tobago', 'Egypt', 'Norway', 'Kazakhstan', 'Malaysia', 'Iraq',
                'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Kuwait', 'Bahrain']]
},
}

# Print classification summary
print("\n" + "="*70)
print("CLASSIFICATION SUMMARY")
print("="*70)
for name, groups in classifications.items():
    print(f"{name}: {len(groups['treatment'])} treatment, {len(groups['control'])} control")

# Thresholds to test
thresholds = [1.0, 0.10, 0.05, 0.01]

# =============================================================================
# 4. RUN ALL REGRESSIONS
# =============================================================================

results_list = []

for class_name, groups in classifications.items():
    
    treatment = groups['treatment']
    control = groups['control']
    
    # Create classification-specific panel
    temp_panel = panel.copy()
    temp_panel['Treatment'] = temp_panel['Country'].isin(treatment).astype(int)
    temp_panel['OVX_x_Treatment'] = temp_panel['OVX_chg'] * temp_panel['Treatment']
    
    for pct in thresholds:
        
        # Filter to threshold
        if pct == 1.0:
            subset = temp_panel.copy()
            threshold_label = 'Full'
        else:
            oil_qtl = temp_panel['OVX_chg'].quantile(1 - pct)
            subset = temp_panel[temp_panel['OVX_chg'] > oil_qtl].copy()
            threshold_label = f'Top {pct*100:.0f}%'
        
        # Skip if too few observations
        if len(subset) < 50:
            continue
        
        # Prepare for linearmodels
        subset['Date'] = pd.to_datetime(subset['Date'])
        reg_data = subset.set_index(['Country', 'Date'])
        
        y = reg_data['CDS_ret']
        
        # Model A: Country FE + explicit controls
        try:
            X_a = reg_data[['OVX_chg', 'OVX_x_Treatment', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
            model_a = PanelOLS(y, X_a, entity_effects=True, time_effects=False, drop_absorbed=True)
            res_a = model_a.fit(cov_type='clustered', cluster_entity=True)
            beta_a = res_a.params['OVX_x_Treatment']
            pval_a = res_a.pvalues['OVX_x_Treatment']
        except:
            beta_a, pval_a = np.nan, np.nan
        
        # Model B: Country FE + Time FE
        try:
            X_b = reg_data[['OVX_x_Treatment']]
            model_b = PanelOLS(y, X_b, entity_effects=True, time_effects=True, drop_absorbed=True)
            res_b = model_b.fit(cov_type='clustered', cluster_entity=True)
            beta_b = res_b.params['OVX_x_Treatment']
            pval_b = res_b.pvalues['OVX_x_Treatment']
        except:
            beta_b, pval_b = np.nan, np.nan
        
        results_list.append({
            'Classification': class_name,
            'Threshold': threshold_label,
            'N': len(subset),
            'N_treatment': subset['Treatment'].sum(),
            'N_control': len(subset) - subset['Treatment'].sum(),
            'Beta_A': beta_a,
            'pval_A': pval_a,
            'Beta_B': beta_b,
            'pval_B': pval_b,
        })

# =============================================================================
# 5. CREATE RESULTS DATAFRAME
# =============================================================================

results_df = pd.DataFrame(results_list)

# =============================================================================
# 6. PRINT RESULTS - MODEL A
# =============================================================================

print("\n" + "="*90)
print("MODEL A: Country FE + Global Controls (VIX, DXY, UST10Y) — EXPANDED SAMPLE (67 countries)")
print("="*90)

pivot_a = results_df.pivot(index='Classification', columns='Threshold', values='Beta_A')
pivot_a = pivot_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

pivot_pval_a = results_df.pivot(index='Classification', columns='Threshold', values='pval_A')
pivot_pval_a = pivot_pval_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

print(f"\n{'Classification':<25} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*75)

for idx in pivot_a.index:
    row_str = f"{idx:<25}"
    for col in pivot_a.columns:
        beta = pivot_a.loc[idx, col]
        pval = pivot_pval_a.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# 7. PRINT RESULTS - MODEL B
# =============================================================================

print("\n" + "="*90)
print("MODEL B: Country FE + Time FE — EXPANDED SAMPLE (67 countries)")
print("="*90)

pivot_b = results_df.pivot(index='Classification', columns='Threshold', values='Beta_B')
pivot_b = pivot_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

pivot_pval_b = results_df.pivot(index='Classification', columns='Threshold', values='pval_B')
pivot_pval_b = pivot_pval_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

print(f"\n{'Classification':<25} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*75)

for idx in pivot_b.index:
    row_str = f"{idx:<25}"
    for col in pivot_b.columns:
        beta = pivot_b.loc[idx, col]
        pval = pivot_pval_b.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# 8. COMPARISON TABLE
# =============================================================================

print("\n" + "="*90)
print("COMPARISON: Oil Exporters vs Placebo Classifications (Top 10% OVX, Model A)")
print("="*90)

top10_results = results_df[results_df['Threshold'] == 'Top 10%'].copy()
top10_results = top10_results.sort_values('Beta_A', ascending=False)

print(f"\n{'Classification':<25} {'Beta':>10} {'p-value':>10} {'N_treat':>10} {'N_ctrl':>10} {'Sig':>6}")
print("-"*75)

for _, row in top10_results.iterrows():
    sig = '***' if row['pval_A'] < 0.01 else '**' if row['pval_A'] < 0.05 else '*' if row['pval_A'] < 0.1 else ''
    print(f"{row['Classification']:<25} {row['Beta_A']:>10.3f} {row['pval_A']:>10.3f} {row['N_treatment']:>10.0f} {row['N_control']:>10.0f} {sig:>6}")

# =============================================================================
# 9. SAVE RESULTS
# =============================================================================

results_df.to_csv('placebo_classification_results_expanded.csv', index=False)
print("\nSaved: placebo_classification_results_expanded.csv")

Full panel shape: (35754, 14)
Countries: 67
Weeks: 537

Countries in panel:
['Abu Dhabi', 'Argentina', 'Australia', 'Austria', 'Bahrain', 'Belgium', 'Brazil', 'Bulgaria', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Dominican Republic', 'Dubai', 'Egypt', 'El Salvador', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Hungary', 'India', 'Indonesia', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Kazakhstan', 'Kuwait', 'Latvia', 'Lithuania', 'Malaysia', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Peru', 'Philippines', 'Poland', 'Qatar', 'Romania', 'Saudi Arabia', 'Serbia', 'Slovakia', 'Slovenia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Thailand', 'Trinidad and Tobago', 'Turkey', 'United Kingdom', 'United States', 'Uruguay', 'Vietnam']

Oil exporters in panel (15): ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Kuwait', 'Bahrain', 'Iraq', 'Colombia', 'Mexico', 'Br